# GraphRAG

## Import packages

In [8]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  bert_score_metric,
)

## Disable warnings

In [9]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [10]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [11]:
app = NeuroRAG(debug=True)
app.compile()

## Evaluate RAG

### Load QA dataset

In [12]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')[:50]
mediqa_df

,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
5,How does the auditory system respond to differ...,The auditory system's response to sound varies...
6,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
7,What brain regions are involved in language pr...,Language processing involves areas in the pref...
8,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...
9,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


### Load cached RAGs responses

In [13]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache[CACHE_KEY].keys())

25

In [14]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []
generation_times = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    start_time = time.perf_counter()
    cache[CACHE_KEY][question] = app.invoke(question)['generation']
    elapsed = time.perf_counter() - start_time
    generation_times.append(elapsed)

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

if generation_times:
  print(f'Generation times (n={len(generation_times)}):')
  print(f'  Mean:   {sum(generation_times) / len(generation_times):.2f}s')
  print(f'  Median: {sorted(generation_times)[len(generation_times) // 2]:.2f}s')
  print(f'  Min:    {min(generation_times):.2f}s')
  print(f'  Max:    {max(generation_times):.2f}s')
  print(f'  Total:  {sum(generation_times):.2f}s')
else:
  print('All answers loaded from cache, no generation times recorded.')

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
print('cos_score', cos_score)
bleu_score = bleu_metric(expected_answers, predicted_answers)
print('bleu_score', bleu_score)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
print('rogue_1_score', rogue_1_score)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
print('rogue_l_score', rogue_l_score)
factscore_score = factscore_metric(expected_answers, predicted_answers)
print('factscore_score', factscore_score)
bert_score = bert_score_metric(expected_answers, predicted_answers)
print('bert_score', bert_score)

16it [00:00, 156.25it/s]

[2026-03-30 22:37:06.069] ---GENERATE STEP-BACK QUERY---
[2026-03-30 22:37:06.950] ---GENERATE SUBQUERIES---
[2026-03-30 22:37:07.315] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-30 22:37:07.981] ---SELECTED SOURCES: ['vectorstore']---
[2026-03-30 22:37:07.981] ---ROUTE QUESTION---
[2026-03-30 22:37:07.981] ---GENERATE HYDE DOCUMENTS---
[2026-03-30 22:37:09.429] ---RETRIEVE FROM VECTOR STORE---
[2026-03-30 22:37:10.855] ---GRADE DOCUMENTS---
[2026-03-30 22:37:10.856] ---RRF RANKED: 13 documents---
[2026-03-30 22:37:10.857] ---BM25 CANDIDATES: 12 documents---
[2026-03-30 22:37:13.584] ---RERANKED TOP DOCS: 5 (scores: [9, 8, 6, 5, 5])---
[2026-03-30 22:37:18.787] ---FINAL DOCUMENTS: 4---
[2026-03-30 22:37:18.788] ---ASSESS GRADED DOCUMENTS---
[2026-03-30 22:37:18.788] ---DECISION: GENERATE---
[2026-03-30 22:37:18.789] ---GENERATE---


16it [00:15, 156.25it/s]

[2026-03-30 22:37:41.261] ---GRADE GENERATION---
[2026-03-30 22:37:42.178] ---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---


26it [00:36,  1.73s/it] 

...

[2026-03-30 23:16:38.399] ---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
[2026-03-30 23:16:38.563] ---DECISION: GENERATION ADDRESSES QUESTION---
[2026-03-30 23:16:38.574] ---GENERATE STEP-BACK QUERY---
[2026-03-30 23:16:38.799] ---GENERATE SUBQUERIES---
[2026-03-30 23:16:39.560] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-30 23:16:40.940] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-03-30 23:16:40.940] ---ROUTE QUESTION---
[2026-03-30 23:16:40.941] ---GENERATE HYDE DOCUMENTS---
[2026-03-30 23:17:46.618][2026-03-30 23:17:46.618] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-03-30 23:17:47.129] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 104857.60 seconds...Too Many Requests, waiting for 104857.60 seconds...

[2026-03-30 23:19:46.619] pub_med_retriever_node timed out
[2026-03-30 23:19:46.621] ---GRADE DOCUMENTS---
[2026-03-30 23:19:46.621] ---RRF RANKED: 7 documents---
[2026-03-30 23:19:46.625] ---BM25 CAND

50it [43:11, 51.84s/it] 

[2026-03-30 23:20:17.883] ---DECISION: GENERATION ADDRESSES QUESTION---
Generation times (n=25):
  Mean:   103.67s
  Median: 86.91s
  Min:    21.04s
  Max:    219.31s
  Total:  2591.64s


cos_score 0.7571215007597384
bleu_score 0.01260344127315622
rogue_1_score 0.30639347850041615
rogue_l_score 0.220953438796559
factscore_score 0.17078261630370797
bert_score 0.12576824426651


Too Many Requests, waiting for 419430.40 seconds...
